# Data Quality Check – Chicago Rideshare Trips

Goal: find problems in the trip data before we use it, and decide how to clean it.

Data used: a sample of 200,000 trips (August 2026) from the City of Chicago Transportation Network Providers dataset. We will rerun this on the full data later.

In [6]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

# find the project root (the folder that has data/ in it), wherever Jupyter started
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("project root:", ROOT)

spark = SparkSession.builder.appName("data-quality").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet(str(ROOT / "data" / "parquet" / "tnp_sample"))
df.printSchema()
print(df.count())

num_cols = ["trip_seconds", "trip_miles", "percent_time_chicago", "percent_distance_chicago",
            "fare", "tip", "additional_charges", "trip_total", "trips_pooled",
            "pickup_centroid_latitude", "pickup_centroid_longitude",
            "dropoff_centroid_latitude", "dropoff_centroid_longitude"]

for c in num_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

project root: /Users/harshshah/Downloads/ride-or-rail
root
 |-- trip_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: string (nullable = true)
 |-- trip_miles: string (nullable = true)
 |-- percent_time_chicago: string (nullable = true)
 |-- percent_distance_chicago: string (nullable = true)
 |-- pickup_census_tract: string (nullable = true)
 |-- dropoff_census_tract: string (nullable = true)
 |-- pickup_community_area: string (nullable = true)
 |-- dropoff_community_area: string (nullable = true)
 |-- fare: string (nullable = true)
 |-- tip: string (nullable = true)
 |-- additional_charges: string (nullable = true)
 |-- trip_total: string (nullable = true)
 |-- shared_trip_authorized: string (nullable = true)
 |-- shared_trip_match: string (nullable = true)
 |-- trips_pooled: string (nullable = true)
 |-- pickup_centroid_latitude: string (nullable = true)
 |-- pickup_centroid_lon

The data loaded correctly: 200,000 rows and 26 columns.

Problem found: almost all number columns (trip_seconds, trip_miles, fare, tip, etc.) are stored as text, not numbers. We need to convert them before doing any math.

In [7]:
df.select(num_cols).describe().show(vertical=True)

[Stage 25:===========================================>              (3 + 1) / 4]

-RECORD 0-----------------------------------------
 summary                    | count               
 trip_seconds               | 200000              
 trip_miles                 | 199999              
 percent_time_chicago       | 199639              
 percent_distance_chicago   | 199630              
 fare                       | 199702              
 tip                        | 199702              
 additional_charges         | 199702              
 trip_total                 | 199702              
 trips_pooled               | 200000              
 pickup_centroid_latitude   | 177439              
 pickup_centroid_longitude  | 177439              
 dropoff_centroid_latitude  | 176898              
 dropoff_centroid_longitude | 176898              
-RECORD 1-----------------------------------------
 summary                    | mean                
 trip_seconds               | 1293.993025         
 trip_miles                 | 8.333653941269665   
 percent_time_chicago       | 0

## Numbers summary

After converting to numbers, a few things look wrong:
- Some trips have 0 seconds, 0 miles, or a $0 fare.
- percent_time_chicago and percent_distance_chicago go above 1.0, even though they should be between 0 and 1.
- Some extreme values: longest trip about 9.6 hours, longest about 270 miles, highest total $518. The average trip is about 21 minutes and 8 miles.

In [8]:
total = df.count()
nulls = df.select([
    F.round(F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2).alias(c)
    for c in df.columns
])
nulls.show(vertical=True)

-RECORD 0---------------------------
 trip_id                    | 0.0   
 trip_start_timestamp       | 0.0   
 trip_end_timestamp         | 0.0   
 trip_seconds               | 0.0   
 trip_miles                 | 0.0   
 percent_time_chicago       | 0.18  
 percent_distance_chicago   | 0.19  
 pickup_census_tract        | 48.11 
 dropoff_census_tract       | 48.1  
 pickup_community_area      | 11.67 
 dropoff_community_area     | 12.06 
 fare                       | 0.15  
 tip                        | 0.15  
 additional_charges         | 0.15  
 trip_total                 | 0.15  
 shared_trip_authorized     | 0.0   
 shared_trip_match          | 0.0   
 trips_pooled               | 0.0   
 pickup_centroid_latitude   | 11.28 
 pickup_centroid_longitude  | 11.28 
 pickup_centroid_location   | 11.28 
 dropoff_centroid_latitude  | 11.55 
 dropoff_centroid_longitude | 11.55 
 dropoff_centroid_location  | 11.55 
 year                       | 0.0   
 month                      | 0.0   



## Missing values

- About 48% of trips are missing the census tract. The city hides tracts for privacy.
- Only about 12% are missing the community area.
- Decision: we will analyze by community area (77 areas) instead of census tract. Using tracts would throw away almost half of the data.

In [9]:
rules = {
    "seconds <= 0":              F.col("trip_seconds") <= 0,
    "miles <= 0":                F.col("trip_miles") <= 0,
    "fare <= 0":                 F.col("fare") <= 0,
    "fare is null":              F.col("fare").isNull(),
    "pct_time > 1":              F.col("percent_time_chicago") > 1,
    "pct_distance > 1":          F.col("percent_distance_chicago") > 1,
    "over 4 hours":              F.col("trip_seconds") > 4 * 3600,
    "over 100 miles":            F.col("trip_miles") > 100,
    "no pickup community area":  F.col("pickup_community_area").isNull(),
    "no dropoff community area": F.col("dropoff_community_area").isNull(),
}

total = df.count()
rows = [(name, df.filter(cond).count()) for name, cond in rules.items()]
summary = spark.createDataFrame(rows, ["rule", "rows_flagged"]) \
               .withColumn("pct", F.round(F.col("rows_flagged") / total * 100, 2))
summary.show(truncate=False)

dupes = df.groupBy("trip_id").count().filter("count > 1").count()
print("duplicate trip_ids:", dupes)

+-------------------------+------------+-----+
|rule                     |rows_flagged|pct  |
+-------------------------+------------+-----+
|seconds <= 0             |2           |0.0  |
|miles <= 0               |11          |0.01 |
|fare <= 0                |9265        |4.63 |
|fare is null             |298         |0.15 |
|pct_time > 1             |465         |0.23 |
|pct_distance > 1         |29368       |14.68|
|over 4 hours             |3           |0.0  |
|over 100 miles           |91          |0.05 |
|no pickup community area |23336       |11.67|
|no dropoff community area|24129       |12.06|
+-------------------------+------------+-----+

duplicate trip_ids: 0


## How many rows each problem affects

- Very small problems: 2 trips with 0 seconds, 11 with 0 miles, 3 longer than 4 hours, 91 longer than 100 miles.
- No duplicate trip IDs.
- Bigger problems that need a closer look before deleting anything:
  - 4.6% of trips have a fare of $0 or less
  - 14.7% have percent_distance_chicago above 1
  - about 12% have no community area

In [10]:
# 1. Are $0 fares real trips? (if trip_total > 0 and miles are short, they're rounding, not errors)
df.filter(F.col("fare") <= 0) \
  .select("trip_miles", "trip_seconds", "trip_total", "tip") \
  .describe().show(vertical=True)

# 2. How far above 1 do the percentages go?
for c in ["percent_distance_chicago", "percent_time_chicago"]:
    print(c,
          "| >1.0:", df.filter(F.col(c) > 1.0).count(),
          "| >1.1:", df.filter(F.col(c) > 1.1).count(),
          "| >1.5:", df.filter(F.col(c) > 1.5).count())

# 3. Do missing community areas line up with trips partly outside Chicago?
df.withColumn("no_area", F.col("pickup_community_area").isNull()) \
  .groupBy("no_area") \
  .agg(F.round(F.avg("percent_distance_chicago"), 3).alias("avg_pct_in_chicago"),
       F.count("*").alias("trips")) \
  .show()

-RECORD 0---------------------------
 summary      | count               
 trip_miles   | 9264                
 trip_seconds | 9265                
 trip_total   | 9265                
 tip          | 9265                
-RECORD 1---------------------------
 summary      | mean                
 trip_miles   | 6.685808506044908   
 trip_seconds | 1028.1664328116567  
 trip_total   | 2.788648677819752   
 tip          | 0.06173772261198057 
-RECORD 2---------------------------
 summary      | stddev              
 trip_miles   | 8.255289604234214   
 trip_seconds | 719.6675651243057   
 trip_total   | 4.899972789937212   
 tip          | 0.4602842787885168  
-RECORD 3---------------------------
 summary      | min                 
 trip_miles   | 0.01                
 trip_seconds | 0.0                 
 trip_total   | 0.0                 
 tip          | 0.0                 
-RECORD 4---------------------------
 summary      | max                 
 trip_miles   | 192.44              
 

## Looking closer at the big problems

**$0 fares:** First we guessed these were small fares rounded down to $0,
because the city rounds fares to the nearest $2.50. That guess was wrong:
these trips average 6.7 miles and 17 minutes, which are normal trips.
So the fare is most likely missing or not reported. The trips themselves are real.

**Percent above 1:** Almost all of these are only slightly above 1.
Only 144 are above 1.1 and only 12 are above 1.5. This looks like small rounding.

**Missing community area:** These trips are only 42% inside Chicago on average,
compared to 90% for other trips. So a missing area means the trip started
or ended outside the city.

## Cleaning rules

| Problem | What we do |
|---|---|
| 0 seconds or 0 miles | Remove |
| Longer than 4 hours or 100 miles | Remove (outliers) |
| Fare $0 or missing | Keep, but don't use for price analysis |
| Percent in Chicago above 1 | Change it to 1.0 |
| No community area | Keep for city totals, but not for area analysis |

Result: we only remove about 0.05% of rows. All other trips are kept, so we don't lose real data.

Next step: run this same notebook on the full dataset.